# DataKwaliteit pipeline: laag `brons_p`, schema `def_complexen_v8`

# Laad packages

In [ ]:
# %pip install nbformat
# %pip install numpy
# %pip install geopandas
# %pip install pandas
# %pip install re
# %pip install duckdb

In [ ]:
import nbformat
import duckdb
from datetime import datetime
import os

#pandas en geopandas voor betere visualisatie bij testen of dingen werken.
import numpy
import geopandas as gpd
import pandas as pd

## Start DuckLake

In [ ]:
%run ./start_ducklake.ipynb

## Laad data

In [ ]:
%run ./laad_data.ipynb

## Laad testen (importeer user-defined functions)

In [ ]:
%run ./datakwaliteit_udf.ipynb

# Voer testen uit

## Losse testen


cpx_code_b/type (beschrijving complexen) komt overeen met ABR

In [ ]:
dq07_kolomwaarde_niet_in_lijst(
    laag_1 = "brons_p",
    schema_1 = "complexen",
    tabel_1 = "def_complexen_v8",
    sleutelkolom_1 = "COMPLEX_ID",
    kolom_1 = "CPX_CODE_B",
    laag_2 = "brons_p",
    schema_2= "abr",
    tabel_2= "complextypen",
    kolom_2 = "label"
)
dq07_kolomwaarde_niet_in_lijst(
    laag_1 = "brons_p",
    schema_1 = "complexen",
    tabel_1 = "archeologische_complexen_rm_2026",
    sleutelkolom_1 = "complex_nummer",
    kolom_1 = "type",
    laag_2 = "brons_p",
    schema_2= "abr",
    tabel_2= "complextypen",
    kolom_2 = "label"
)

cpx_code_N (abr codes complexen) komt overeen met ABR

In [ ]:
dq07_kolomwaarde_niet_in_lijst(
    laag_1 = "brons_p",
    schema_1 = "complexen",
    tabel_1 = "def_complexen_v8",
    sleutelkolom_1 = "COMPLEX_ID",
    kolom_1 = "CPX_CODE_N",
    laag_2 = "brons_p",
    schema_2= "abr",
    tabel_2= "complextypen",
    kolom_2 = "codeAbr"
)

## Datakwaliteitsdimensies

In [ ]:
D8TC_geometrie_valide(
    laag_1= "brons_p",
    schema_1= "complexen",
    tabel_1= "def_complexen_v8",
    sleutelkolom_1 = "COMPLEX_ID",
    geometriekolom_1 = "geom"
)

# Bekijk de data

## Selecteer een DataSet tabel

In [ ]:
con.sql("SELECT * FROM brons_p.complexen.def_complexen_v8 WHERE geom_d8tC = False;").fetchdf()

In [ ]:
con.sql("SELECT DISTINCT type, type_abr_code FROM brons_p.complexen.archeologische_complexen_rm_2026 WHERE type_dq07 == False").fetchdf()

In [ ]:
con.sql("SELECT omschrijving FROM brons_p.complexen.archeologische_complexen_rm_2026 WHERE type == 'wal/omwalling'").fetchdf()


In [ ]:
con.sql("SELECT * FROM brons_p.abr.complextypen LIMIT 5;").fetchdf()

In [ ]:
con.sql(f"""
SELECT
    t2.id,
    t2.location_name AS name_in_table_2,
    r.location_name  AS matched_reference_name,
    CASE
        WHEN r.location_name IS NULL THEN 'NO SPATIAL MATCH'
        WHEN lower(trim(t2.location_name)) = lower(trim(r.location_name))
            THEN 'MATCH'
        ELSE 'NAME MISMATCH'
    END AS check_result
FROM brons_p.complexen.archeologische_complexen_rm_2026 AS t2
LEFT JOIN ref_table AS r
    ON ST_Within(t2.geom, r.geom);
""")